# Phase 5-C1 — Research Authorization (administration only)

No strategy execution, ResearchOnlyRunner, Holdout read or Batch use. Start with **VALIDATE_ONLY** in a fresh Runtime.
All commands use an isolated **Python 3.11** environment, even if the Colab kernel is newer.
A reviewed exact code SHA is required; branch HEAD is never used as a code pin.

**Run All never approves.** Cell 5 only installs a button handler. Approval requires the lead's live GitHub identity,
an independent authorization PR, explicit settings, a separate checkbox, typing the authorization ID and clicking the button.
A local approval is not effective until its exact files are synchronized to the PR and reviewed/merged by the lead.
The schema/code delivery must first reach main. No cell commits, pushes or merges.


## Cell 1 — Human Settings

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json, os, re, sys, shutil, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
REPOSITORY = 'https://github.com/hh4832/strategy-research-agent.git'
CODE_BRANCH = 'fix/phase5c1-unsigned-preview-compat'
CODE_SHA = 'b6188d57630761665ca6e466c837a3d7668f89a9'  # exact reviewed code pin; supplied with the delivery
AUTHORIZATION_ID = 'RA-IBKR_TSM-SMA520_P5C1-0001'
GITHUB_PR_URL = ''  # actual separate authorization PR; never this code PR or an old approval URL
RESEARCH_LEAD = 'hh4832'
VALIDITY_DAYS = 7
APPROVAL_MODE = 'VALIDATE_ONLY'  # VALIDATE_ONLY / PREPARE_APPROVAL / APPROVE
CONFIRM_CODE_REVIEW = False  # personally review the exact code version and set True
CONFIRM_AUTHORIZATION = False  # still insufficient alone: manual Cell 5 UI click is also required
EXPECTED_APPROVED_RECORD_SHA256 = ''  # only for verifying a prior session's real approved event after merge
ARCHIVE_ROOT = Path('/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c1/20261111_230118_690789')
DRIVE_PARENT = Path('/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c1_authorization')

assert REPOSITORY == 'https://github.com/hh4832/strategy-research-agent.git'
assert re.fullmatch('[0-9a-f]{40}', CODE_SHA), 'Enter the exact reviewed code SHA from the delivery'
assert CONFIRM_CODE_REVIEW, 'Personally review and confirm the code pin first'
assert APPROVAL_MODE in ('VALIDATE_ONLY','PREPARE_APPROVAL','APPROVE')
assert AUTHORIZATION_ID == 'RA-IBKR_TSM-SMA520_P5C1-0001' and RESEARCH_LEAD == 'hh4832'
assert type(VALIDITY_DAYS) is int and VALIDITY_DAYS == 7
RUN_ID = datetime.now(ZoneInfo('Asia/Taipei')).strftime('%Y%m%d_%H%M%S_%f')+'_'+CODE_SHA[:12]
SESSION = Path('/content/phase5c1_authorization')/RUN_ID
SESSION.mkdir(parents=True,exist_ok=False)
CODE_ROOT = SESSION/'code'
SETTINGS = dict(mode=APPROVAL_MODE,confirm=CONFIRM_AUTHORIZATION,authorization_id=AUTHORIZATION_ID,
    lead=RESEARCH_LEAD,validity_days=VALIDITY_DAYS,pr_url=GITHUB_PR_URL,archive=str(ARCHIVE_ROOT),
    code_sha=CODE_SHA,code_branch=CODE_BRANCH,work_root=str(SESSION),expected_record_sha256=EXPECTED_APPROVED_RECORD_SHA256)
SETTINGS_PATH=SESSION/'settings.json'
SETTINGS_PATH.write_text(json.dumps(SETTINGS,indent=2))  # no credentials in this file
VALIDATION_OUT=SESSION/'validation'
APPROVAL_OUT=SESSION/'human_approval'  # created only by the explicit manual click
print('Mode:',APPROVAL_MODE,'| Code pin:',CODE_SHA,'| No authorization has been written.')


## Cell 2 — Isolated Environment & Repository Verification

In [ ]:
def command(args, **kwargs):
    result=subprocess.run([str(x) for x in args],check=True,text=True,capture_output=True,**kwargs)
    return result.stdout.strip()
command(['git','clone','--branch',CODE_BRANCH,'--single-branch',REPOSITORY,CODE_ROOT])
command(['git','-C',CODE_ROOT,'checkout','--detach',CODE_SHA])
assert command(['git','-C',CODE_ROOT,'remote','get-url','origin'])==REPOSITORY
assert command(['git','-C',CODE_ROOT,'rev-parse','HEAD'])==CODE_SHA
assert command(['git','-C',CODE_ROOT,'status','--porcelain'])==''
command(['git','-C',CODE_ROOT,'merge-base','--is-ancestor',CODE_SHA,'origin/'+CODE_BRANCH])

# Provision Python 3.11 locally without altering the Colab kernel or another project.
interpreter=shutil.which('python3.11')
if not interpreter:
    bootstrap=SESSION/'bootstrap'
    command([sys.executable,'-m','pip','install','--target',bootstrap,'uv==0.9.5'])
    uv_env=dict(os.environ,PYTHONPATH=str(bootstrap),UV_PYTHON_INSTALL_DIR=str(SESSION/'managed_python'),
        UV_CACHE_DIR=str(SESSION/'uv_cache'),UV_PYTHON_BIN_DIR=str(SESSION/'python_bin'))
    command([sys.executable,'-m','uv','python','install','3.11'],env=uv_env)
    interpreter=command([sys.executable,'-m','uv','python','find','--managed-python','3.11'],env=uv_env)
command([interpreter,'-m','venv',SESSION/'venv'])
PYTHON=SESSION/'venv/bin/python'
assert command([PYTHON,'-c','import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")'])=='3.11'
command([PYTHON,'-m','pip','install','-e',CODE_ROOT])
command([PYTHON,'-m','pip','check'])
schema=json.loads((CODE_ROOT/'schemas/RESEARCH_AUTHORIZATION_SCHEMA.json').read_text())
assert schema['$schema']=='https://json-schema.org/draft/2020-12/schema'
assert {'dated_schedule_path','dated_schedule_sha256'} <= set(schema['$defs']['approve']['properties']['costs']['properties'])
assert command(['git','-C',CODE_ROOT,'status','--porcelain'])==''
print('Python:',command([PYTHON,'--version']),'| Commit:',CODE_SHA,'| Schema: research-only-authorization-v1 (dated-pair compatible)')

def run_management(action,out,*,human_confirmation='',live_github=False):
    env=dict(os.environ)
    env.pop('GITHUB_TOKEN',None);env.pop('GH_TOKEN',None)
    if live_github:
        # Token is used only in this subprocess environment; never saved, printed or passed in arguments.
        env['GITHUB_TOKEN']=userdata.get('GITHUB_TOKEN')
        assert env['GITHUB_TOKEN'], 'Set your own GITHUB_TOKEN in Colab Secrets; do not paste it into this Notebook'
    args=[PYTHON,'-m','strategy_agent.readiness.c1_authorization',action,'--settings',SETTINGS_PATH,'--out',out]
    if human_confirmation: args+=['--human-confirmation',human_confirmation]
    return command(args,env=env,cwd=CODE_ROOT)


## Cell 3 — Proposal / Archive / Governance Validation

In [ ]:
# Reads whitelisted archived JSON and official fee source documents only; never opens snapshot Parquet.
action='prepare' if APPROVAL_MODE=='PREPARE_APPROVAL' else 'validate'
try:
    print(run_management(action,VALIDATION_OUT))
except subprocess.CalledProcessError as error:
    # No token is present in this validation command. Fail closed before any approval UI can become usable.
    if not (VALIDATION_OUT/'authorization_validation.json').is_file():
        raise RuntimeError(error.stderr or error.stdout) from None
    print('Validation failed; keeping the failure report for Drive audit. Approval is locked.')
VALIDATION=json.loads((VALIDATION_OUT/'authorization_validation.json').read_text())
for check,status in VALIDATION['checks'].items(): print(f'{status:4} | {check}')
PR_VERIFIED=False
if VALIDATION['status']=='PASS' and GITHUB_PR_URL:
    try:
        print(run_management('github',VALIDATION_OUT,live_github=True))
        PR_VERIFIED=True
    except Exception:
        print('GitHub verification failed; approval remains locked. Review github_verification.json if available.')
else:
    print('PR verification: NOT_PROVIDED; approval remains blocked. This is allowed for validation/preparation.')


## Cell 4 — Asia/Taipei Validity Preview (not approval)

In [ ]:
print(json.dumps(VALIDATION['validity_preview'],indent=2))
print('Preview only. Actual issued_at is chosen afresh at the manual approval click; validity is seven days.')
print('PREPARE_APPROVAL emits an unsigned preview, not an APPROVE event.')


## Cell 5 — Independent Human Approval Gate

Run All only displays this UI. To approve, the lead must select APPROVE and CONFIRM_AUTHORIZATION=True in Cell 1
in a fresh session, pass all validations and live PR checks, tick this separate confirmation, type the exact ID,
and personally click. A failure requires inspection/new session; no retry is automated.
The admin tool alone appends the validated chained record in an isolated clone of the actual authorization PR.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
human_checkbox=widgets.Checkbox(value=False,description='I personally approve this exact one-Batch scope')
human_id=widgets.Text(value='',description='Authorization ID:')
approve_button=widgets.Button(description='Personally approve authorization',button_style='danger',
    disabled=not (APPROVAL_MODE=='APPROVE' and CONFIRM_AUTHORIZATION and GITHUB_PR_URL and PR_VERIFIED and VALIDATION['status']=='PASS'))
approval_messages=widgets.Output()

def _human_approve(_button):
    with approval_messages:
        assert APPROVAL_MODE=='APPROVE' and CONFIRM_AUTHORIZATION is True and PR_VERIFIED
        assert human_checkbox.value is True and human_id.value==AUTHORIZATION_ID
        approve_button.disabled=True  # one explicit attempt per isolated session
        print(run_management('approve',APPROVAL_OUT,human_confirmation=human_id.value,live_github=True))
        print('LOCAL_APPROVED_PENDING_PR_SYNC: not effective; no commit, push or merge was performed.')
        print('Run Cell 6 then Cell 7 now to verify and back up the exact review + PR synchronization files.')

approve_button.on_click(_human_approve)  # registering this handler never calls it
# Programmatic Notebook execution has no approval callback invocation.
display(human_checkbox,human_id,approve_button,approval_messages)


## Cell 6 — Verify / Synchronize (no automatic Git operations that write remotely)

A local event is not effective. Cell 7 preserves exact proposed files and a patch. The human must personally
commit/push them to the **same separate authorization PR**, review its diff and merge it after review.
No local Python/Git/VS Code installation is needed: open the printed GitHub web-edit links, copy the exact saved file contents, and personally commit each change to the same PR branch in the browser. Review the complete diff before merging. Never substitute an old PR URL. After merge, set VERIFY_MERGED=True here and rerun this cell; in a later Runtime,
also enter the genuine approved record SHA in Cell 1. Verification reads the merged hash chain and compares the
exact record, validity, unchanged gates and unused budget. It does not prepare or execute B3-C.


In [ ]:
VERIFY_MERGED = False  # manually enable only after the separate authorization PR was actually merged
if VERIFY_MERGED:
    verification_out=APPROVAL_OUT if (APPROVAL_OUT/'authorization_review.json').is_file() else VALIDATION_OUT
    print(run_management('verify-merged',verification_out,live_github=True))
elif (APPROVAL_OUT/'authorization_review.json').is_file():
    REVIEW=json.loads((APPROVAL_OUT/'authorization_review.json').read_text())
    print(REVIEW['status'],'| Effective:',REVIEW['effective'])
    print('Exact record SHA:',REVIEW['authorization']['record_sha256'])
    print('Human PR synchronization files:',APPROVAL_OUT/'human_pr_sync')
    from urllib.parse import quote
    branch=quote(REVIEW['github_identity']['head_branch'],safe='')
    for relative in ('governance/proposals/'+AUTHORIZATION_ID+'.json','governance/research_authorizations.jsonl'):
        print('Human GitHub web edit (use exact saved file contents):','https://github.com/hh4832/strategy-research-agent/edit/'+branch+'/'+relative)
    print('After your personal commit/push, review/merge the separate PR and rerun Cell 6 with VERIFY_MERGED=True.')
else:
    print('NOT_APPROVED: validation/preview only; no event exists from this Notebook run.')


## Cell 7 — Immutable Google Drive Evidence Output

In [ ]:
from hashlib import sha256

def archive_evidence():
    stamp=datetime.now(ZoneInfo('Asia/Taipei')).strftime('%Y%m%d_%H%M%S_%f')+'_'+CODE_SHA[:12]
    destination=DRIVE_PARENT/stamp
    destination.mkdir(parents=True,exist_ok=False)
    # Explicit whitelist only. Never archive settings, environment variables, git config or tokens.
    allowed=('authorization_validation.json','authorization_review.json','proposal.UNSIGNED.PREVIEW.json',
        'github_verification.json','merged_verification.json','human_pr_sync.patch',
        'human_pr_sync/governance/proposals/RA-IBKR_TSM-SMA520_P5C1-0001.json',
        'human_pr_sync/governance/research_authorizations.jsonl')
    hashes={}
    for label,source_root in (('validation',VALIDATION_OUT),('human_approval',APPROVAL_OUT)):
        for relative in allowed:
            source=source_root/relative
            if not source.is_file(): continue
            assert source.resolve().is_relative_to(source_root.resolve())
            data=source.read_bytes();before=sha256(data).hexdigest()
            target=destination/label/relative;target.parent.mkdir(parents=True,exist_ok=True)
            with target.open('xb') as f: f.write(data)
            assert before==sha256(source.read_bytes()).hexdigest()==sha256(target.read_bytes()).hexdigest()
            hashes[str(target.relative_to(destination))]=before
    status='NOT_APPROVED'
    merged=[p for p in (APPROVAL_OUT/'merged_verification.json',VALIDATION_OUT/'merged_verification.json') if p.is_file()]
    if merged:status=json.loads(merged[0].read_text())['status']
    elif (APPROVAL_OUT/'authorization_review.json').is_file():status='LOCAL_APPROVED_PENDING_PR_SYNC'
    info=f'Code commit: {CODE_SHA}\nPython: {command([PYTHON,"--version"])}\nGitHub PR: {GITHUB_PR_URL or "NOT_PROVIDED"}\nMode: {APPROVAL_MODE}\nAuthorization status: {status}\nStrategy execution: NOT_EXECUTED\nBatch use by this Notebook: 0\n'
    with (destination/'run_info.txt').open('x') as f:f.write(info)
    hashes['run_info.txt']=sha256((destination/'run_info.txt').read_bytes()).hexdigest()
    with (destination/'ARCHIVE_INDEX.json').open('x') as f:json.dump({'files_sha256':hashes,'code_commit':CODE_SHA,'authorization_status':status,'timezone':'Asia/Taipei'},f,indent=2)
    print(info);print('Drive evidence archive:',destination)
    return destination

DRIVE_EVIDENCE_ARCHIVE=archive_evidence()


## Stop boundary

This Notebook never generates BIND_AUTHORIZATION/APPROVE_TASK or invokes an execution mode.
After the exact authorization is effective and verified, use the separately reviewed C1/B3-C preparation workflow.
The research Notebook's older Agent code pin must not be assumed compatible: use a reviewed compatible code version
and explicitly pin its governance checkout. Formal Market Gate remains NO_GO, Holdout stays NO_ACCESS, max_batches=1.
